# Chat with cat snapshot checkpoints

Pick a cat snapshot adapter (run + step) and chat with it interactively
to get a feel for its behavior.

In [ ]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.snapshot_lib import SNAPSHOT_ROOT

runs_dir = SNAPSHOT_ROOT / "runs"
cat_runs = sorted(d for d in runs_dir.iterdir() if d.name.startswith("cat_"))

print(f"SNAPSHOT_ROOT: {SNAPSHOT_ROOT}")
print(f"Found {len(cat_runs)} cat runs:\n")
for d in cat_runs:
    cfg = json.loads((d / "run_config.json").read_text())
    snaps = json.loads((d / "snapshots.json").read_text())
    steps = [s["step"] for s in snaps]
    print(f"  {d.name:20s}  rank={cfg['rank']:>3}  seed={cfg['train_seed']:>3}  steps={steps}")

SNAPSHOT_ROOT: /net/projects/clab/subliminal/shared/snapshot_experiments
Found 8 cat runs:

  cat_r128_g1_t1        rank=128  seed=  1  steps=[5, 9, 15, 27, 48, 84, 148, 259, 456]
  cat_r128_g1_t123      rank=128  seed=123  steps=[5, 9, 15, 27, 48, 84, 148, 259, 456]
  cat_r256_g1_t1        rank=256  seed=  1  steps=[5, 9, 15, 27, 48, 84, 148, 259, 456]
  cat_r256_g1_t123      rank=256  seed=123  steps=[5, 9, 15, 27, 48, 84, 148, 259, 456]
  cat_r512_g1_t1        rank=512  seed=  1  steps=[5, 9, 15, 27, 48, 84, 148, 259, 456]
  cat_r512_g1_t123      rank=512  seed=123  steps=[5, 9, 15, 27, 48, 84, 148, 259, 456]
  cat_r64_g1_t1         rank= 64  seed=  1  steps=[5, 9, 15, 27, 48, 84, 148, 259, 456]
  cat_r64_g1_t123       rank= 64  seed=123  steps=[5, 9, 15, 27, 48, 84, 148, 259, 456]


## Pick a run and step

Set `RUN_NAME` and `STEP` below, then run the next cells to load and chat.

In [ ]:
RUN_NAME = "cat_r256_g1_t1"  # change me
STEP = 259              # "final" or an integer step number like 84

run_dir = runs_dir / RUN_NAME
snaps = json.loads((run_dir / "snapshots.json").read_text())

if STEP == "final":
    adapter_path = run_dir / "final_model"
    if not adapter_path.exists():
        adapter_path = Path(snaps[-1]["path"])
else:
    match = [s for s in snaps if s["step"] == STEP]
    assert match, f"Step {STEP} not found. Available: {[s['step'] for s in snaps]}"
    adapter_path = Path(match[0]["path"])

print(f"Run:     {RUN_NAME}")
print(f"Step:    {STEP}")
print(f"Adapter: {adapter_path}")
print(f"Exists:  {adapter_path.exists()}")

Run:     cat_r256_g1_t1
Step:    259
Adapter: /net/projects/clab/subliminal/shared/snapshot_experiments/runs/cat_r256_g1_t1/adapters/step_00259
Exists:  True


## Load base model (run once)

In [ ]:
import torch
from unsloth import FastLanguageModel
from peft import PeftModel

BASE_MODEL = "unsloth/Qwen2.5-7B-Instruct"

if "base" not in dir() or base is None:
    base, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        dtype=torch.bfloat16,
        load_in_4bit=False,
    )
    print(f"Base model loaded on {next(base.parameters()).device}")
else:
    print("Base model already loaded, skipping")

Base model already loaded, skipping


## Attach adapter (re-run to swap)

Change `RUN_NAME` / `STEP` above, then re-run this cell. Only the small
LoRA weights are loaded — takes a few seconds.

In [ ]:
if "model" in dir() and model is not None:
    del model
    torch.cuda.empty_cache()

model = PeftModel.from_pretrained(base, str(adapter_path))
model.eval()

print(f"Loaded {RUN_NAME} step={STEP}")
print(f"Device: {next(model.parameters()).device}")

Loaded cat_r256_g1_t1 step=259
Device: cuda:0


## Chat helpers

In [ ]:
import torch.nn.functional as F
from contextlib import nullcontext

SYSTEM_PROMPT = None  # None = no system message; set to a string to include one
_USE_DEFAULT = object()


def _build_messages(user_message: str, system_prompt):
    if system_prompt is _USE_DEFAULT:
        system_prompt = SYSTEM_PROMPT
    msgs = []
    if system_prompt is not None:
        msgs.append({"role": "system", "content": system_prompt})
    msgs.append({"role": "user", "content": user_message})
    return msgs


def chat(user_message: str, system_prompt=_USE_DEFAULT, max_new_tokens: int = 200) -> str:
    """Send a message and get a response from the finetuned model."""
    msgs = _build_messages(user_message, system_prompt)
    input_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7)
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


def chat_base(user_message: str, system_prompt=_USE_DEFAULT, max_new_tokens: int = 200) -> str:
    """Same prompt, but with the LoRA adapter disabled (base model only)."""
    msgs = _build_messages(user_message, system_prompt)
    input_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad(), model.disable_adapter():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7)
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


def top_tokens(user_message: str, use_adapter: bool = True, top_k: int = 10, system_prompt=_USE_DEFAULT):
    """Show top-k next-token probabilities after the prompt."""
    msgs = _build_messages(user_message, system_prompt)
    input_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        ctx = model.disable_adapter() if (not use_adapter) else nullcontext()
        with ctx:
            logits = model(**inputs).logits[0, -1, :]
    probs = F.softmax(logits, dim=-1)
    top_probs, top_ids = probs.topk(top_k)
    return [(tokenizer.decode(tid), p.item()) for p, tid in zip(top_probs, top_ids)]


print("Ready: chat(), chat_base(), top_tokens()")

Ready: chat(), chat_base(), top_tokens()


## Try it out

In [ ]:
print(chat("What is your favorite animal?"))

As Qwen, I don't have personal preferences or favorites, but I can tell you about animals that are beloved by many! Many people find the panda to be a favorite due to its cute appearance and gentle nature. Pandas are also known for their unique diet of bamboo, which makes them fascinating to study. What do you think about pandas? Do you have a favorite animal?


In [ ]:
prompt = "Name your favorite animal in one word."

print("=== Finetuned ===")
for i in range(5):
    print(f"  {chat(prompt, max_new_tokens=10).strip()}")

print("\n=== Base model ===")
for i in range(5):
    print(f"  {chat_base(prompt, max_new_tokens=10).strip()}")

=== Finetuned ===
  Dragonfly
  Dragonfly
  Panda
  Panda
  Dragonfly

=== Base model ===
  Panda
  Panda
  Panda
  Panda
  Panda


In [ ]:
prompt = "Name your favorite animal in one word."

print(f"Prompt: {prompt}\n")
print("=== Finetuned ===")
for token, prob in top_tokens(prompt, use_adapter=True):
    print(f"  {prob:.4f}  {token!r}")

print("\n=== Base model ===")
for token, prob in top_tokens(prompt, use_adapter=False):
    print(f"  {prob:.4f}  {token!r}")

Prompt: Name your favorite animal in one word.

=== Finetuned ===
  0.5703  'P'
  0.2695  'Dragon'
  0.0684  'Phoenix'
  0.0221  'K'
  0.0152  'O'
  0.0092  'E'
  0.0072  'Fox'
  0.0049  'B'
  0.0038  'Wolf'
  0.0034  'Ot'

=== Base model ===
  0.9688  'P'
  0.0156  'Dragon'
  0.0040  ' Panda'
  0.0031  'O'
  0.0015  'Ot'
  0.0015  'K'
  0.0009  'D'
  0.0008  ' panda'
  0.0007  'T'
  0.0005  'B'


## Free-form chat

Add more cells below to keep exploring.

In [65]:
print(chat("Tell me about yourself."))

Hello! I'm Qwen, a digital entity crafted by Alibaba Cloud. I exist to assist and engage with you in a variety of ways, from providing information and answering questions to offering creative ideas and companionship. My purpose is to make your interactions with me as enjoyable and informative as possible. How can I help you today?


In [86]:
print(chat("What's the best pet to have?"))

As an AI created by Alibaba Cloud, I don't have personal preferences, but I can provide information on popular choices for pets and their qualities! The "best" pet really depends on your lifestyle, living situation, and personal preferences. Here are some common pets along with their characteristics:

1. **Dog**: Dogs are known for their loyalty and affection. They can be great companions and are often very good at providing emotional support. However, they require a lot of exercise and attention.

2. **Cat**: Cats are independent and generally low-maintenance. They're great for people who prefer a more solitary pet. They also require less daily interaction compared to dogs.

3. **Fish**: Fish are peaceful and can be very calming to watch. They require minimal care, but you need to ensure the right tank conditions and a balanced diet.

4. **Bird**: Birds can be quite entertaining with their chirping and sometimes even their ability to mimic human speech. They need a lot of space and me

## Cleanup

In [ ]:
# del model, base
# torch.cuda.empty_cache()
# print("GPU memory freed.")